# MedGemma ICD-10 coding on pediatric encounter notes — Colab Runner (T4 free tier)

**The question, as asked:** given a real encounter note with the Assessment
removed, does `google/medgemma-4b-it` output the right ICD-10 codes? Precision
and recall, exact code match.

**This is a different task from every other notebook in this repo.** The MDACE
and MIMIC runners ask the model to *extract phrases* and score phrase overlap.
This one asks for the **code** and scores exact code match — `B08.5` either
equals `B08.5` or it does not. There is no fuzzy matching ladder here and none
is wanted.

---

### Read this before you read any number it prints

**16 gold codes, 4 notes.** One code is **6.25 recall points**. That is wider
than most differences anyone would want to report. Whatever comes out of this is
a **spot check**, not a benchmark, and it cannot carry a decision on its own. If
a real figure is needed, the ask is ~50 notes.

**Three input variants, one prompt.** They differ only in how much of the note
the model is shown, so every difference between the three numbers is caused by
the text that was removed and not by a prompt change.

| variant | what it is | what it is for |
|---|---|---|
| `full` | nothing removed — the `DX` lines are still on the page | **Harness check, not a result.** If the model cannot return `B08.5` while `DX 1: B08.5` is printed in front of it, the bug is in the prompt or the parser. Run it first. |
| `assessment_cut` | Assessment block removed | **What was asked for.** |
| `leakage_cut` | Assessment **and** Problem List removed | **The honest number.** |

**Why `leakage_cut` exists.** Removing the Assessment is not enough. Note 26819
prints `- J30.2 OTHER SEASONAL ALLERGIC RHINITIS` and
`- L20.9 ATOPIC DERMATITIS, UNSPECIFIED` in its Problem List — the gold code
strings themselves — and notes 55688 and 112976 print three more gold
descriptions word for word. Under `assessment_cut` the model can copy those
instead of reasoning. Both numbers get reported.

**Runtime:** 12 model calls, each note in a single prompt. Well under 30 minutes
on a free T4 — this is a short run, unlike the recall benchmark.

## 1. Confirm the T4 GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU. Set Runtime -> Change runtime type -> T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))

## 2. Install dependencies

Shorter than the recall runner's list: no `sentence-transformers`, because there
is no embedding-based matching level. Exact code match needs no model but the
one under test.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes tqdm huggingface_hub

## 3. Hugging Face login (gated model)

Accept the license at https://huggingface.co/google/medgemma-4b-it, then paste a
token from https://huggingface.co/settings/tokens. Read via `getpass`, so it is
never stored in the notebook.

In [ ]:
from getpass import getpass
from huggingface_hub import login
login(getpass('HF token (input hidden): '))

## 4. Get the project code — and re-run this cell to pull updates

Code only. The repo is public and contains **no** patient data — the notes are
uploaded by hand in step 6 and never committed.

**This cell is idempotent: run it again any time to pull the latest code.** It
clones on first use and hard-resets to the remote branch afterwards.

> **The notebook in your browser is a separate copy from the repo.** Running
> this cell updates the *files on the VM*, never the notebook you are reading.
> If a cell you were told about is missing, close the tab and reopen the link.

In [ ]:
import os

REPO = '/content/medgemma-ner-eval'
BRANCH = 'billing-icd-eval'
URL = 'https://github.com/shifat514/medgemma-ner-eval.git'

if os.path.isdir(os.path.join(REPO, '.git')):
    !git -C {REPO} fetch -q origin {BRANCH}
    !git -C {REPO} checkout -q {BRANCH}
    !git -C {REPO} reset -q --hard origin/{BRANCH}
    print('pulled')
else:
    !git clone -q {URL} {REPO}
    !git -C {REPO} checkout -q {BRANCH}
    print('cloned')

%cd {REPO}
!git log -1 --oneline
!git status --short --branch | head -1

## 5. Mount Drive for run output

Colab's disk is wiped when the runtime recycles; a Drive folder is not.
`BILLING_OUTPUT_DIR` redirects the per-note state there, so a disconnect costs
only the call in flight.

| file | contents | shareable |
|---|---|---|
| `per_note.jsonl` | counts and ICD codes | yes — a code is a catalogue entry, it names no patient |
| `replies.jsonl` | raw model output | **no** — it can quote the note |

At 12 calls a disconnect is unlikely to cost much, but the cache is also what
lets you re-run one variant without re-running the other two.

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

OUT = '/content/drive/MyDrive/billing-icd-outputs'
os.makedirs(OUT, exist_ok=True)
os.environ['BILLING_OUTPUT_DIR'] = OUT
print('run output ->', OUT)
print('existing run dirs:', os.listdir(OUT) or '(none yet)')

## 5b. What is cached on Drive, and clearing it

Runs are keyed on model, token cap and **prompt hash**. Anything already cached
is **skipped** on a re-run — that is what survives a disconnect, and also what
reports old numbers in about a second if the directory is stale.

Editing the prompt changes the hash and therefore the directory, so an edited
prompt cannot silently replay old results. Editing the *sample* does not. If you
rebuild the sample locally and re-upload it, clear the cache here.

Set `CLEAR` to a directory name to delete it. There is no undo.

In [ ]:
import glob, json, os, shutil

OUT = os.environ['BILLING_OUTPUT_DIR']

CLEAR = None   # e.g. 'medgemma-4b-it_tok512_pa1b2c3d4'

for d in sorted(glob.glob(os.path.join(OUT, '*'))):
    if not os.path.isdir(d):
        continue
    pn = os.path.join(d, 'per_note.jsonl')
    n = sum(1 for _ in open(pn)) if os.path.exists(pn) else 0
    print(f'{os.path.basename(d):<45} {n:>3} (note, variant) pairs cached')

if CLEAR:
    target = os.path.join(OUT, CLEAR)
    assert os.path.isdir(target), f'no such run dir: {target}'
    shutil.rmtree(target)
    print('\ndeleted', target)

## 6. Upload the sample file BY HAND

**The PDFs never come to Colab.** They are parsed on the machine that holds
them, with `make billing-sample`, which writes one small JSONL. That file is
what you upload here.

On the local machine:

```bash
make billing-sample          # -> data/samples/billing_sample.jsonl
```

Then pick that file in the dialog below. It is **not** in git and never will be
— it carries note text, patient names and dates of birth.

The cell checks what arrived: 4 notes, 17 gold DX lines, 16 unique codes, and
the leak counts per variant (`full` 16, `assessment_cut` 2, `leakage_cut` 0).
Those numbers have right answers. If they do not match, the parse is wrong and
nothing downstream is worth running.

In [ ]:
import json, os, shutil
from google.colab import files

DEST = '/content/medgemma-ner-eval/data/samples/billing_sample.jsonl'
os.makedirs(os.path.dirname(DEST), exist_ok=True)

if os.path.exists(DEST):
    print('already present:', DEST)
else:
    up = files.upload()
    name = next(iter(up))
    shutil.move(name, DEST)
    print('saved ->', DEST)

recs = [json.loads(l) for l in open(DEST) if l.strip()]
lines = sum(r['n_gold_lines'] for r in recs)
uniq = sum(r['n_gold_unique'] for r in recs)

print(f'\nnotes                {len(recs)}          (expect 4)')
print(f'gold DX lines        {lines}         (expect 17)')
print(f'gold unique codes    {uniq}         (expect 16)')
print()
for v in ('full', 'assessment_cut', 'leakage_cut'):
    leaked = sum(len(r['leaked_codes'][v]) for r in recs)
    words = sum(r['n_words'][v] for r in recs)
    print(f'  {v:<16} {words:>5} words   gold codes still visible: {leaked}')
print('  (expect 16, 2, 0)')
print()
for r in recs:
    print(f"  {r['note_id']:<8} {r['visit_kind']:<10} {', '.join(r['gold_codes'])}")

## 7. Harness check #1 — the oracle. No GPU, ~1 second.

Replays gold back through the **real** parser and the **real** scorer with no
model involved. Every variant must read `1.0000 / 1.0000`.

This is not ceremony. On the two previous branches an equivalent check caught
real bugs twice, and both times the broken version produced a plausible-looking
number instead of an error. Run it before spending GPU time.

In [ ]:
!python -m src.evaluate_billing --oracle

## 8. Harness check #2 — `full`, with the model. 4 calls.

The note **with the DX lines still in it**. The model is being asked to read
codes off the page, so this should score high.

**This is not a result.** It is the check that separates "the model cannot code
this note" from "the prompt or the parser is broken". If this comes back low,
stop — the two numbers that matter would be measuring the harness, not the
model.

In [ ]:
!python -m src.evaluate_billing --variant full --dump-replies

## 9. The run — all three variants. 12 calls.

Re-uses the four calls from step 8, so this adds eight. Resumable: re-run the
cell after a disconnect and it picks up where it stopped.

In [ ]:
!python -m src.evaluate_billing --dump-replies

## 10. Read the answer key in full

16 codes fits on one screen, and reading it is strictly more informative than
reading its mean. The two things to look for:

- **`Z68.51` / `Z68.52`** are BMI-percentile codes. The note prints
  `BMI: 17.8 (24 %ile)` and the code depends on knowing that the 24th percentile
  maps to `Z68.52`. That is a lookup table, not reading comprehension — a miss
  there is a different failure from missing a diagnosis. 3 of the 16 gold codes
  are these.
- **`Z00.121`** needs "this is a well visit *with abnormal findings*", which is
  a judgement about the visit type rather than about a condition.

If the misses are concentrated in those four, the model is reading the notes
fine and failing at code-book mechanics — which is a very different report from
the model missing the influenza.

In [ ]:
!python -m src.evaluate_billing --score-only

## 11. Take the numbers home

`results/billing_icd_*.json` is aggregate only — counts, rates, and the run
metadata. No note text, no patient identifiers. That is the file to paste into
a message.

`replies.jsonl` on Drive holds the raw model output and **can quote the note**.
Leave it on Drive; do not commit it and do not paste it.

In [ ]:
import json, glob

for p in sorted(glob.glob('results/billing_icd_*.json')):
    print('=' * 70)
    print(p)
    print('=' * 70)
    print(json.dumps(json.load(open(p)), indent=2))